In [ ]:
import os
import pandas as pd
from morphing import generate_morphed_datasets
from extract_metafeatures import process_metafeatures
from landmarkers import extract_landmarkers_from_folder

BASE_DIR = os.getcwd()
DATASET_A = os.path.join(BASE_DIR, "data", "dataset_source_A.csv")
DATASET_B = os.path.join(BASE_DIR, "data", "dataset_source_B.csv")
MORPHED_DATASETS_DIR = os.path.join(BASE_DIR, "output", "morphed_datasets")
METAFEATURES_DIR = os.path.join(BASE_DIR, "output", "metafeatures")
LANDMARKERS_DIR = os.path.join(BASE_DIR, "output", "landmarkers")

for folder in [MORPHED_DATASETS_DIR, METAFEATURES_DIR, LANDMARKERS_DIR]:
    os.makedirs(folder, exist_ok=True)

## Step 1 - Dataset Morphing

In [ ]:
print("--- [STEP 1/3] Executing Dataset Morphing Pipeline ---")

# Define trajectory seeds (change/add seeds to generate alternative trajectories)
SEEDS = [1, 42]
SWAP_RATIO = 0.1  # 10% progressive row swaps per step

all_morphed_files = []

for seed in SEEDS:
    print(f"\n>>> Running Morphing Trajectory for SEED = {seed} <<<")
    files = generate_morphed_datasets(
        dataset_a_path=DATASET_A,
        dataset_b_path=DATASET_B,
        output_folder=MORPHED_DATASETS_DIR,
        swap_ratio=SWAP_RATIO,
        prefix="morph",
        seed=seed
    )
    all_morphed_files.extend(files)

print(f"\n Finished Morphing! Generated {len(all_morphed_files)} synthetic datasets across {len(SEEDS)} trajectory run(s).")

## Step 2 - Metafeatures Extraction

In [ ]:
print("--- [STEP 2/3] Starting Metafeature Extraction for All Strategies ---")

# Define all strategies to execute
STRATEGIES = ["A", "B", "C", "D", "E"]
extracted_files = []

for strategy in STRATEGIES:
    print(f"\n Executing Metafeature Extraction Strategy: '{strategy}'...")
    process_metafeatures(
        input_folder=MORPHED_DATASETS_DIR,
        output_folder=METAFEATURES_DIR,
        strategy=strategy
    )
    output_file = os.path.join(METAFEATURES_DIR, f"metafeatures_{strategy}_final.csv")
    extracted_files.append(output_file)
    print(f" Strategy '{strategy}' completed! Saved to:\n {output_file}")

print("\n All metafeature strategies ('A', 'B', 'C', 'D', 'E') extracted successfully!")

In [ ]:
# Optional: Combine all extracted strategy CSV files into a single master DataFrame
meta_dfs = []
for file_path in extracted_files:
    if os.path.exists(file_path):
        df_strat = pd.read_csv(file_path)
        meta_dfs.append(df_strat)

if meta_dfs:
    # Merge strategy dataframes on the 'dataset' column
    from functools import reduce
    df_combined_meta = reduce(lambda left, right: pd.merge(left, right, on='dataset', how='outer'), meta_dfs)
    master_meta_path = os.path.join(METAFEATURES_DIR, "all_metafeatures_combined.csv")
    df_combined_meta.to_csv(master_meta_path, index=False)
    print(f"\n Master combined metafeatures file created at:\n {master_meta_path}")

## Step 3 - Landmarkers Extraction

In [ ]:
print("--- [STEP 3/3] Evaluating Landmarkers on Intermediate Datasets ---")

extract_landmarkers_from_folder(
    datasets_dir=MORPHED_DATASETS_DIR,
    output_dir=LANDMARKERS_DIR
)

landmarker_results_path = os.path.join(LANDMARKERS_DIR, "landmarkers_final.csv")
print(f"\n Landmarkers evaluated successfully! Saved to:\n{landmarker_results_path}")